# Improved Mamba for 4D Navier-Stokes (Kaggle, unattended, ~4-5 h)

MLP vs Mamba vs space-time Mamba (spatial PointMamba scan across Gaussians) vs space-time Mamba with velocity integration; 5 flows, 10% observed.

**Settings:** Accelerator = **GPU T4 x2** (or P100), **Internet = On** → **Save Version → Save & Run All (Commit)**.
Output: **`thesis1_mamba4d_results.zip`** (Output tab).


In [ ]:
REPO_URL = "https://github.com/muhammadhussainqureshi/thesis-gs"   # change after transferring the repo
CONFIG = "thesis1_mamba4d"    # configs/<CONFIG>.json
QUICK = False               # True = few-minute smoke test with tiny settings

import os, subprocess, sys, glob, zipfile, shutil, torch
WORK = "/kaggle/working"
CODE = "/kaggle/working/thesis-gs"
DATA = f"{WORK}/data"
OUT = f"{WORK}/{CONFIG}"
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - set Accelerator to GPU")
if os.path.isdir(CODE):
    shutil.rmtree(CODE)
subprocess.run(["git", "clone", "--depth", "1", REPO_URL, CODE], check=True)
assert os.path.isfile(f"{CODE}/configs/{CONFIG}.json"), "config not found - is the latest code pushed?"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "lmdb", "tifffile"], check=True)
print(subprocess.run(["git", "-C", CODE, "log", "-1", "--oneline"], capture_output=True, text=True).stdout)


In [ ]:
VOL = "/tmp/cytopacq"   # not used by the 4D runs
os.makedirs(VOL, exist_ok=True)


In [ ]:
extra = ["--quick", "--transmitters", "8"] if QUICK else []
cmd = [sys.executable, f"{CODE}/bin/run_experiments.py", f"{CODE}/configs/{CONFIG}.json", "--out", OUT,
       "--data_dir", DATA, "--volumes", VOL] + extra
subprocess.run(cmd)        # failures are logged per run; the notebook keeps going


In [ ]:
import pandas as pd
from IPython.display import Image, display
pd.set_option("display.width", 200)
for f in sorted(glob.glob(f"{OUT}/**/summary.csv", recursive=True)):
    print("\n=====", os.path.relpath(os.path.dirname(f), OUT))
    print(pd.read_csv(f).to_string(index=False))
    png = os.path.join(os.path.dirname(f), "reconstructions.png")
    if os.path.exists(png):
        display(Image(filename=png))
for f in glob.glob(f"{OUT}/**/*.pt", recursive=True):   # keep the zip small
    os.remove(f)
os.makedirs(OUT, exist_ok=True)
shutil.make_archive(f"{WORK}/{CONFIG}_results", "zip", OUT)
print(f"\nDownload {CONFIG}_results.zip from the Output tab.")
